<a href="https://colab.research.google.com/github/MarceCorreal2/Robots-NT/blob/main/Procesamiento_Indicadores_Backtest_V6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Este cuaderno limpia y organiza los datos del resumen del strategy analyzer de los robots y debe incluir los indicadores en la
# tabla robots y procesar la calificación

## Procesamiento de Indicadores de Backtest

El objetivo de este cuaderno es tomar los datos crudos de los resúmenes del *strategy analyzer* de tus robots, limpiarlos y extraer los indicadores clave mencionados. Una vez procesados, los datos se guardarán en un formato limpio para futuros análisis.

In [183]:
# Celda 1 — Importes y configuración

import pandas as pd
import os
import re
from datetime import datetime
import pytz # Para manejar zonas horarias, movido aquí
import gspread # Para interactuar con Google Sheets
from google.colab import auth # Para la autenticación en Colab

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

In [184]:
# Celda 1.1 - Mas instalaciones

!pip install pytz

In [185]:
# Celda 2 — Conectar Drive

#from google.colab import drive
#drive.mount('/content/drive')

In [186]:
#Celda 2.1 - Conectar Colab con Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [187]:
# Celda 2.2 - Autenticación con Google Drive y gspread (usando credenciales explícitas)

import google.auth # Importar google.auth

# Autenticar con Google para permitir el acceso a Google Sheets y Drive
# Usamos authenticate_user() para el flujo interactivo de Colab
auth.authenticate_user()

# Obtener las credenciales por defecto del usuario autenticado en Colab
creds, project = google.auth.default()

# Usar estas credenciales para inicializar el cliente de gspread
gc = gspread.Client(auth=creds)

print("Autenticación con Google Sheets completada usando credenciales de Google Colab.")

Autenticación con Google Sheets completada usando credenciales de Google Colab.


In [188]:
# Celda 3 - Cargar la Tabla de Calificación de Bots desde Google Sheets

# Nombre de tu Google Sheet (asegúrate de que coincida exactamente)
BOT_RATING_SHEET_NAME = 'Tabla de Calificación Bots'

try:
    # Abrir el Google Sheet por su nombre
    worksheet = gc.open(BOT_RATING_SHEET_NAME).sheet1

    # Obtener todos los datos como un DataFrame de pandas
    # `get_as_dataframe` lee la primera fila como encabezados automáticamente
    rating_table_df = pd.DataFrame(worksheet.get_all_records())

    print(f"Google Sheet '{BOT_RATING_SHEET_NAME}' cargado exitosamente.")
    print("Primeras 5 filas de la tabla de calificación:")
    display(rating_table_df.head())

    print("Columnas y tipos de datos de la tabla de calificación:")
    display(rating_table_df.info())

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: El Google Sheet '{BOT_RATING_SHEET_NAME}' no se encontró.")
    print("Por favor, verifica que el nombre sea exacto y que tengas permisos de acceso.")
    rating_table_df = pd.DataFrame()
except Exception as e:
    print(f"Ocurrió un error al cargar el Google Sheet: {e}")
    rating_table_df = pd.DataFrame()

Google Sheet 'Tabla de Calificación Bots' cargado exitosamente.
Primeras 5 filas de la tabla de calificación:


,Indicador,Descripción,Fórmula,Peso,Calificación 0,Calificación 2,Calificación 4,Calificación 5,Calificación 8,Calificación 10,
0,PF,Eficiencia de ganancias vs pérdidas,Ganancia Bruta ÷ Pérdida Bruta,25%,<1.00,1.00–1.19,1.20–1.39,1.40–1.59,1.60–1.99,≥2.00,1
1,PayoffRatio,Capacidad de recuperar el Drawdown,Avg Win ÷ Avg Loss,25%,< 0.75,0.75 – 1.19,1.2-1.39,1.40-1.79,1.80-2.49,≥2.5,
2,Percent profitable,Relación ganancia/pérdida por operación,Wins ÷ Total Trades,20%,<30%,30–39%,40–49%,50–54%,55–59%,≥60%,
3,Sharpe ratio,Porcentaje de operaciones ganadoras,Retorno ajustado por volatilidad,10%,<0,0–0.49,0.50–0.99,1.00–1.49,1.50–1.99,≥2.00,
4,Recovery Factor,Rentabilidad vs variabilidad,Net Profit ÷ Max DD,20%,<0.50,0.50–0.99,1.00–1.49,1.50–2.49,2.50–3.99,≥4.00,


Columnas y tipos de datos de la tabla de calificación:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Indicador         8 non-null      object
 1   Descripción       8 non-null      object
 2   Fórmula           8 non-null      object
 3   Peso              8 non-null      object
 4   Calificación 0    8 non-null      object
 5   Calificación   2  8 non-null      object
 6   Calificación 4    8 non-null      object
 7   Calificación  5   8 non-null      object
 8   Calificación 8    8 non-null      object
 9   Calificación 10   8 non-null      object
 10                    8 non-null      object
dtypes: object(11)
memory usage: 836.0+ bytes


None

In [189]:
#Celda 3.1 import math # Se añadió para float('inf')

rating_criteria = {}

# Pesos predefinidos se construirán dinámicamente desde rating_table_df
default_weights = {}

# --- NUEVO: Construir default_weights dinámicamente desde rating_table_df ---
for idx, row in rating_table_df.iterrows():
    indicator_name = str(row['Indicador']).strip()
    # Asegurarse de que 'Peso' sea una columna válida antes de intentar acceder a ella
    if 'Peso' in row and pd.notna(row['Peso']):
        peso_str = str(row['Peso']).strip().replace('%', '').replace(',', '.') # Reemplazar coma por punto para float
        if indicator_name and peso_str:
            try:
                weight = float(peso_str)
                default_weights[indicator_name] = weight
            except ValueError:
                print(f"Advertencia: No se pudo parsear el peso para '{indicator_name}': '{row['Peso']}'. Saltando.")
    else:
        print(f"Advertencia: La columna 'Peso' no está disponible o es nula para el indicador '{indicator_name}'. Saltando.")
# --- FIN NUEVO ---


# Función auxiliar para parsear rangos o valores categóricos
def parse_criteria_value(s):
    s = str(s).replace('%', '').strip()  # Eliminar el signo de porcentaje y espacios en blanco

    if s == '' or s.lower() == 'nan':  # Manejar cadenas vacías o NaN
        return None

    # Manejar rangos explícitos '[lower, upper]'
    if s.startswith('[') and s.endswith(']'):
        try:
            parts = s.replace('[', '').replace(']', '').split(',')
            # Corregido el SyntaxError: 'parts[0']' a 'parts[0]'
            lower = float(parts[0]) if parts[0].lower() != '-inf' else -float('inf')
            upper = float(parts[1]) if parts[1].lower() != 'inf' else float('inf')
            return (lower, upper)
        except ValueError:
            return None

    # Manejar rangos 'X–Y' o 'X-Y'
    if '–' in s:  # Guion largo
        parts = s.split('–')
        try:
            lower = float(parts[0])
            upper = float(parts[1])
            return (lower, upper)
        except ValueError:
            return None
    elif '-' in s and s.count('-') == 1 and s[0] != '-':  # Guion normal, asegurando que no sea un número negativo
        parts = s.split('-')
        if len(parts) == 2 and parts[0].replace('.', '', 1).isdigit() and parts[1].replace('.', '', 1).isdigit():
            try:
                lower = float(parts[0])
                upper = float(parts[1])
                return (lower, upper)
            except ValueError:
                pass  # Fallback si no es un rango válido con guion

    # Manejar '<X'
    if s.startswith('<'):
        try:
            value = float(s[1:])
            return (-float('inf'), value)
        except ValueError:
            return None

    # Manejar '≥X' o '>=X'
    if s.startswith('≥') or s.startswith('>='):
        try:
            value = float(s[1:].strip()) if s.startswith('≥') else float(s[2:].strip())
            return (value, float('inf'))
        except ValueError:
            return None

    # Manejar '>X'
    if s.startswith('>'):
        try:
            value = float(s[1:])
            return (value, float('inf'))
        except ValueError:
            return None

    # Manejar categorías explícitas, mapeándolas a los tres niveles principales
    s_lower = s.lower()
    if s_lower == 'excelente':
        return {'type': 'category', 'value': 'Excelente'}
    elif s_lower in ['muy bueno', 'bueno', 'aceptable']: # Mapear 'Muy Bueno' y 'Bueno' a 'Aceptable'
        return {'type': 'category', 'value': 'Aceptable'}
    elif s_lower in ['regular', 'malo']: # Mapear 'Regular' a 'Malo'
        return {'type': 'category', 'value': 'Malo'}

    # Intentar parsear como un solo número
    try:
        val = float(s)
        return (val, val)  # Un solo valor se considera un punto fijo
    except ValueError:
        return None  # No se pudo parsear

# Helper function to check if a string appears to be a valid criteria string
def is_valid_criteria_string(s):
    s_cleaned = str(s).replace('%', '').strip() # Limpiar el string antes de verificar la validez
    if s_cleaned == '' or s_cleaned.lower() == 'nan':
        return False
    # Comprobar patrones comunes de rango/desigualdad en la cadena limpia
    if any(op in s_cleaned for op in ['[', ']', '–', '-', '<', '>', '≥', '>=']):
        return True
    # Corregido el SyntaxError: Eliminado el '+' extra al final de la línea
    s_lower = s_cleaned.lower()
    if s_lower in ['excelente', 'aceptable', 'malo', 'bueno', 'regular', 'muy bueno']:
        return True
    # Considerar un string como '2.0' como un criterio válido para coincidencia de un solo punto
    try:
        float(s_cleaned)
        return True
    except ValueError:
        pass
    return False

# Columnas que contienen los criterios de calificación y sus puntuaciones correspondientes
# Las claves de este diccionario ahora coinciden EXACTAMENTE con los nombres de columna del DataFrame
SCORE_COLUMNS = {
    'Calificación 0': 0,
    'Calificación   2': 20,
    'Calificación 4': 40,
    'Calificación  5': 60, # Observado como 'Calificación  5' en rating_table_df.info()
    'Calificación 8': 80,
    'Calificación 10': 100
}

# Iterate through each indicator for which we expect to find criteria
for indicator_name, weight in default_weights.items():
    # Find rows in rating_table_df that match the indicator name
    # Usamos .str.strip() para manejar posibles espacios en blanco invisibles
    candidate_rows = rating_table_df[rating_table_df['Indicador'].str.strip() == indicator_name.strip()].copy()

    selected_row = None
    max_valid_criteria_count = 0

    # Prioritize rows that have the most valid criteria strings in the score columns
    for idx, row in candidate_rows.iterrows():
        current_valid_criteria_count = 0
        for col_full_name in SCORE_COLUMNS.keys(): # Ahora iteramos sobre los nombres de columna completos
            if is_valid_criteria_string(row.get(col_full_name, '')):
                current_valid_criteria_count += 1

        if current_valid_criteria_count > max_valid_criteria_count: # Corrección: el nombre de la variable era incorrecto
            max_valid_criteria_count = current_valid_criteria_count
            selected_row = row

    if selected_row is None or max_valid_criteria_count == 0: # Fallback if no criteria rows were found, or if all had 0 valid criteria strings.
        print(f"Advertencia: No se encontraron criterios válidos para el indicador '{indicator_name}'. Saltando.")
        continue

    # Store all parsed ranges and their corresponding scores
    parsed_ranges_with_scores = []
    for col_full_name, score_value in SCORE_COLUMNS.items(): # Iteramos sobre los nombres de columna completos
        criteria_str = str(selected_row.get(col_full_name, '')).strip()
        if is_valid_criteria_string(criteria_str):
            parsed_range = parse_criteria_value(criteria_str)
            if parsed_range is not None:
                parsed_ranges_with_scores.append({'score': score_value, 'range': parsed_range})

    # Sort by score in ascending order so that when we iterate, we can find the highest matching score
    parsed_ranges_with_scores.sort(key=lambda x: x['score'])

    if not parsed_ranges_with_scores:
        print(f"Advertencia: No se pudieron parsear rangos válidos para el indicador '{indicator_name}'. Saltando.")
        continue

    rating_criteria[indicator_name] = {
        'weight': weight,
        'ranges': parsed_ranges_with_scores
    }

# Imprimir los criterios procesados para verificación
print("Criterios de Calificación Procesados (`rating_criteria`):")
for indicator, criteria in rating_criteria.items():
    print(f"- {indicator}:")
    print(f"  Ponderación: {criteria['weight']}% ")
    for r in criteria['ranges']:
        print(f"    Puntos: {r['score']}, Rango: {r['range']}")

Criterios de Calificación Procesados (`rating_criteria`):
- PF:
  Ponderación: 25.0% 
    Puntos: 0, Rango: (-inf, 1.0)
    Puntos: 20, Rango: (1.0, 1.19)
    Puntos: 40, Rango: (1.2, 1.39)
    Puntos: 60, Rango: (1.4, 1.59)
    Puntos: 80, Rango: (1.6, 1.99)
    Puntos: 100, Rango: (2.0, inf)
- PayoffRatio:
  Ponderación: 25.0% 
    Puntos: 0, Rango: (-inf, 0.75)
    Puntos: 20, Rango: (0.75, 1.19)
    Puntos: 40, Rango: (1.2, 1.39)
    Puntos: 60, Rango: (1.4, 1.79)
    Puntos: 80, Rango: (1.8, 2.49)
    Puntos: 100, Rango: (2.5, inf)
- Percent profitable:
  Ponderación: 20.0% 
    Puntos: 0, Rango: (-inf, 30.0)
    Puntos: 20, Rango: (30.0, 39.0)
    Puntos: 40, Rango: (40.0, 49.0)
    Puntos: 60, Rango: (50.0, 54.0)
    Puntos: 80, Rango: (55.0, 59.0)
    Puntos: 100, Rango: (60.0, inf)
- Sharpe ratio:
  Ponderación: 10.0% 
    Puntos: 0, Rango: (-inf, 0.0)
    Puntos: 20, Rango: (0.0, 0.49)
    Puntos: 40, Rango: (0.5, 0.99)
    Puntos: 60, Rango: (1.0, 1.49)
    Puntos: 80, Rango

In [190]:
# Celda 4 — Definir Variables

bot_name = 'RB001' # <-- Ajustado para que coincida con el nombre del archivo
test_number = 'T001' # <--- Corregido para que coincida con el archivo disponible
instrument = 'MNQ'

print(f"Bot Name: {bot_name}")
print(f"Test Number: {test_number}")
print(f"Instrument: {instrument}")

Bot Name: RB001
Test Number: T001
Instrument: MNQ


In [191]:
#Celda 5 - Inicializa comentario

import ipywidgets as widgets
from IPython.display import display

# Initialize with an empty string or retrieve existing comment if possible
# For simplicity, we'll start with an empty string, assuming the user will input new comments.
# The logic in Celda 9 will handle retrieving existing comments if this input is left blank.
user_input_comment_widget = widgets.Textarea(
    value='',
    placeholder='Escribe tu comentario aquí para el robot y test actuales...',
    description='Comentario:',
    disabled=False,
    layout=widgets.Layout(width='auto', height='80px') # Adjust width to be flexible
)

def on_value_change(change):
    global comment_for_current_run
    comment_for_current_run = change.new

user_input_comment_widget.observe(on_value_change, names='value')

# Initialize global variable
comment_for_current_run = user_input_comment_widget.value

display(user_input_comment_widget)
print("Haz clic fuera del campo de texto o presiona Enter/Tab para guardar el comentario.")

Textarea(value='', description='Comentario:', layout=Layout(height='80px', width='auto'), placeholder='Escribe…

Haz clic fuera del campo de texto o presiona Enter/Tab para guardar el comentario.


In [192]:
# Celda 5.1 - Verificación del Comentario para la Ejecución Actual (opcional)

# Muestra el comentario que se usará para la ejecución actual, según lo ingresado en el widget.
# Esto te ayuda a confirmar que el valor está siendo capturado correctamente.
print(f"Comentario capturado para la ejecución actual: {comment_for_current_run if 'comment_for_current_run' in globals() else 'Ninguno (variable no definida o vacía)'}")

Comentario capturado para la ejecución actual: 


In [193]:
# Celda 6 — Definir Rutas dinámicas

RAW_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/'
CLEAN_DATA_PATH = '/content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/'

print(f"Ruta de Datos Crudos: {RAW_DATA_PATH}")
print(f"Ruta de Datos Limpios: {CLEAN_DATA_PATH}")

Ruta de Datos Crudos: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/
Ruta de Datos Limpios: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/


In [194]:
#Celda 6.1 - Construir el nombre del archivo dinámicamente usando las variables definidas en Celda 3
# Ajustado para que el nombre del archivo sea `RB001_MNQ_A_T003.csv`

sample_file_name = f'SA_{bot_name}_{instrument}_A_{test_number}.csv' # Corrected filename to include '_A_'
sample_file_path = os.path.join(RAW_DATA_PATH, sample_file_name)

print(f"Cargando archivo de ejemplo: {sample_file_path}")

try:
    # Read the first few lines to understand the structure and find the actual data start
    raw_lines = []
    with open(sample_file_path, 'r', encoding='latin1') as f:
        for _ in range(30): # Read up to 30 lines to cover potential header length
            line = f.readline()
            if not line: # EOF
                break
            raw_lines.append(line.strip())

    print("\nPrimeras 20 líneas del archivo crudo para inspección:")
    for i, line in enumerate(raw_lines[:20]):
        print(f"Línea {i+1}: {line}")

    # Try to find the line that indicates the start of the actual performance metrics
    # Common indicators like 'Total net profit' usually appear at the start of data section.
    data_start_row = -1
    for i, line in enumerate(raw_lines):
        if 'Total net profit' in line:
            data_start_row = i
            break

    if data_start_row != -1:
        print(f"\nIdentificado el inicio de los datos de indicadores en la línea (0-index): {data_start_row}")
        # Read the CSV again, skipping lines up to the identified data start
        # We set header=None because the first column will contain the indicator names,
        # and the subsequent columns are values (e.g., All trades, Long trades, Short trades).
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

        # Assuming the first column is the indicator name and the next are its values
        # We need to clean up the column names based on the context. Sample: Performance;All trades;Long trades;Short trades
        # Let's just display the raw parsed DataFrame for now.
        print("\nDataFrame de indicadores procesado (primeras 5 filas):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del DataFrame procesado:")
        print(sample_df.columns.tolist())
    else:
        print("\nNo se pudo identificar el inicio de los datos de indicadores ('Total net profit' no encontrado). Se muestra la lectura inicial sin procesar.")
        # Fallback if specific data start not found, try reading with just separator
        sample_df = pd.read_csv(sample_file_path, encoding='latin1', sep=';')
        print("\nPrimeras 5 filas del archivo de ejemplo (lectura básica):")
        print(sample_df.head().to_markdown(index=False, numalign="left", stralign="left"))
        print("\nColumnas del archivo de ejemplo (lectura básica):")
        print(sample_df.columns.tolist())

except FileNotFoundError:
    print(f"Error: El archivo '{sample_file_name}' no se encontró en la ruta '{RAW_DATA_PATH}'. Por favor, verifica la ruta y el nombre del archivo.")
    print("\nArchivos CSV disponibles en la carpeta de datos crudos:")
    csv_files = [f for f in os.listdir(RAW_DATA_PATH) if f.endswith('.csv')]
    if csv_files:
        for f in csv_files:
            print(f"- {f}")
    else:
        print("No se encontraron archivos CSV en esta ruta.")
except Exception as e:
    print(f"Error al leer el archivo CSV: {e}")

Cargando archivo de ejemplo: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Crudos/SA_RB001_MNQ_A_T001.csv

Primeras 20 líneas del archivo crudo para inspección:
Línea 1: Performance;All trades;Long trades;Short trades;
Línea 2: Total net profit;$ 5680,50;$ 5680,50;$ 0,00;
Línea 3: Gross profit;$ 30262,50;$ 30262,50;$ 0,00;
Línea 4: Gross loss;-$ 24582,00;-$ 24582,00;$ 0,00;
Línea 5: Commission;$ 1045,00;$ 1045,00;$ 0,00;
Línea 6: Profit factor;1,23;1,23;1,00;
Línea 7: Max drawdown;-$ 6111,80;-$ 6111,80;$ 0,00;
Línea 8: Sharpe ratio;0,26;0,26;1,00;
Línea 9: Sortino ratio;0,60;0,60;1,00;
Línea 10: Ulcer index;0,05;0,05;0,00;
Línea 11: R squared;0,39;0,39;0,00;
Línea 12: Total Fees;$ 0,00;$ 0,00;$ 0,00;
Línea 13: Probability;12,52 %;12,52 %;0,00 %;
Línea 14: ;;;;
Línea 15: Start date;1/01/2026;;;
Línea 16: Start time;12:00 AM;;;
Línea 17: End date;30/06/2026;;;
Línea 18: End time;12:00 AM;;;
Línea 19: ;;;;
Línea 20: Total # of trades;550;550;0;

Identificado el inici

In [195]:
import re
from datetime import datetime

# Celda #7 - Limpieza de datos (imports y opciones de pandas movidos a Celda 1)

# Lista para almacenar los DataFrames de indicadores de cada archivo
all_indicators_list = []

# --- AÑADIDO: Asegurar que sample_file_name sea el correcto para esta celda ---
sample_file_name = f'SA_{bot_name}_{instrument}_A_{test_number}.csv' # Corrected filename to include '_A_'
# --- FIN AÑADIDO ---

# Función para limpiar y convertir valores numéricos
def clean_numeric_value(value):
    if isinstance(value, str):
        # CORRECCIÓN: Usar '.replace(' ', '')' en lugar de 're.escape(' ')'
        value = value.replace('$', '').replace(' ', '').replace('%', '').replace(',', '.')
        if value == '' or value == '-':
            return None
        try:
            return float(value)
        except ValueError:
            return value
    return value

# Usar el sample_file_name y sample_file_path ya definidos en Celda 5
# para procesar solo el archivo deseado.
print(f"Procesando el archivo especificado: {sample_file_name}")

# full_file_path ya está definido en Celda 5 y es el que queremos procesar
# Construimos full_file_path nuevamente aquí para asegurar que sea el correcto
# si Celda 5 no se ejecuta justo antes.
full_file_path = os.path.join(RAW_DATA_PATH, sample_file_name)

try:
    # --- Parsing robusto (similar a Celda 5) ---
    raw_lines = []
    with open(full_file_path, 'r', encoding='latin1') as f:
        for _ in range(30): # Read up to 30 lines to find data start
            line = f.readline()
            if not line: break
            raw_lines.append(line.strip())

    data_start_row = -1
    for i, line in enumerate(raw_lines):
        if 'Total net profit' in line:
            data_start_row = i
            break

    if data_start_row == -1:
        print(f"Advertencia: No se encontró el inicio de datos para {sample_file_name}. No se procesará este archivo.")
    else:
        temp_df = pd.read_csv(full_file_path, encoding='latin1', sep=';', skiprows=data_start_row, header=None)

        # --- Limpieza y extracción (similar a Celda 6)---
        # Aplicar nombres de columna iniciales y establecer índice
        temp_df.columns = ['Performance', 'All trades', 'Long trades', 'Short trades', 'Extra_Column']
        temp_df = temp_df.drop(columns=['Extra_Column'])
        temp_df['Performance'] = temp_df['Performance'].str.strip()
        temp_df = temp_df.set_index('Performance')

        # Aplicar limpieza a valores numéricos
        for col in ['All trades', 'Long trades', 'Short trades']:
            temp_df[col] = temp_df[col].apply(clean_numeric_value)

        def get_indicator_value(df, indicator_name):
            try:
                return df.loc[indicator_name.strip(), 'All trades']
            except KeyError:
                return None

        # Extraer los indicadores solicitados
        TotalNetProfit = get_indicator_value(temp_df, 'Total net profit')
        PF = get_indicator_value(temp_df, 'Profit factor')
        # CAMBIO: Extraer 'Percent profitable' directamente del raw
        PercentProfitable_value = get_indicator_value(temp_df, 'Percent profitable')
        MaxDrawdown = get_indicator_value(temp_df, 'Max drawdown')
        TotalTrades = get_indicator_value(temp_df, 'Total # of trades')
        Winners = get_indicator_value(temp_df, 'Number of winning trades')
        GrossProfit = get_indicator_value(temp_df, 'Gross profit')
        GrossLoss = get_indicator_value(temp_df, 'Gross loss')
        AvgWinTrade = get_indicator_value(temp_df, 'Avg winning trade')
        AvgLossTrade = get_indicator_value(temp_df, 'Avg losing trade')
        SharpeRatio = get_indicator_value(temp_df, 'Sharpe ratio')

        # PayoffRatio calculation (no WR needed for Probability now)
        PayoffRatio = (AvgWinTrade / abs(AvgLossTrade)) if AvgWinTrade is not None and AvgLossTrade is not None and AvgLossTrade != 0 else None

        # Calculate RecoveryFactor as Total Net Profit / abs(Max Drawdown)
        RecoveryFactor = (TotalNetProfit / abs(MaxDrawdown)) if TotalNetProfit is not None and MaxDrawdown is not None and MaxDrawdown != 0 else None

        # Extracción de fechas y cálculo de Net Profit/Mes
        FechaInicio = None
        FechaFin = None
        for line in raw_lines:
            if 'Start date' in line:
                match = re.search(r'Start date;(\d{1,2}/\d{1,2}/\d{4});', line)
                if match:
                    FechaInicio = datetime.strptime(match.group(1), '%d/%m/%Y')
            elif 'End date' in line:
                match = re.search(r'End date;(\d{1,2}/\d{1,2}/\d{4});', line)
                if match:
                    FechaFin = datetime.strptime(match.group(1), '%d/%m/%Y')

        NumMonths = None
        NetProfitPerMonth = None
        if TotalNetProfit is not None and FechaInicio is not None and FechaFin is not None:
            delta = FechaFin - FechaInicio
            if delta.days > 0:
                NumMonths = delta.days / 30.44
                if NumMonths > 0:
                    NetProfitPerMonth = TotalNetProfit / NumMonths

        # Debug print: Verificar los valores antes de crear el diccionario
        print(f"\nValores a usar para Robot Base: {bot_name}, Test ID: {test_number}, Instrumento: {instrument}")

        # Crear un diccionario con los indicadores para este archivo
        file_indicators = {
            'Archivo': sample_file_name,
            'Robot Base': bot_name,
            'Test ID': test_number,
            'Instrumento': instrument,
            'Total net profit': TotalNetProfit,
            'PF': PF,
            'Probability': PercentProfitable_value, # CAMBIO: Usar 'Percent profitable' renombrado a 'Probability'
            'Max drawdown': MaxDrawdown,
            'Recovery Factor': RecoveryFactor,
            'PayoffRatio': PayoffRatio,
            'Sharpe Ratio': SharpeRatio,
            'Total # of trades': TotalTrades,
            '# Meses': NumMonths,
            'Net Profit/Mes': NetProfitPerMonth,
            'Fecha-Inicio': FechaInicio.strftime('%Y-%m-%d') if FechaInicio else None,
            'Fecha-Fin': FechaFin.strftime('%Y-%m-%d') if FechaFin else None,
            'Avg Win': AvgWinTrade,
            'Avg Loss': AvgLossTrade
        }
        all_indicators_list.append(file_indicators)

        # --- Nuevo código para guardar el archivo limpio individual sin subdirectorios dinámicos ---
        individual_df = pd.DataFrame([file_indicators])
        base_file_name = os.path.splitext(sample_file_name)[0]
        cleaned_individual_file_name = f"{base_file_name}_Limpio.csv"

        dynamic_output_dir = CLEAN_DATA_PATH
        os.makedirs(dynamic_output_dir, exist_ok=True)

        individual_output_path = os.path.join(dynamic_output_dir, cleaned_individual_file_name)
        individual_df.to_csv(individual_output_path, index=False)
        print(f"Indicadores individuales guardados en: {individual_output_path}")
        # --- Fin del nuevo código ---

except FileNotFoundError:
    print(f"Error: El archivo '{sample_file_name}' no se encontró en la ruta '{RAW_DATA_PATH}'. Por favor, verifica la ruta y el nombre del archivo.")
except Exception as e:
    print(f"Error al procesar el archivo {sample_file_name}: {e}")

# Convertir la lista de diccionarios a un DataFrame consolidado
if all_indicators_list:
    consolidated_df = pd.DataFrame(all_indicators_list)
    print("\n--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---")
    display(consolidated_df.head())

    # No guardar el DataFrame consolidado aquí ya que la Celda 9 se encarga de la tabla maestra.
elif 'consolidated_df' not in globals(): # Solo imprimir si no se pudo crear consolidated_df
    print("No se pudieron procesar indicadores de ningún archivo.")

Procesando el archivo especificado: SA_RB001_MNQ_A_T001.csv

Valores a usar para Robot Base: RB001, Test ID: T001, Instrumento: MNQ
Indicadores individuales guardados en: /content/drive/MyDrive/Robots/Mis Bots/Análisis Estadístico/Datos Limpios/SA_RB001_MNQ_A_T001_Limpio.csv

--- DataFrame Consolidado de Indicadores (primeras 5 filas): ---


,Archivo,Robot Base,Test ID,Instrumento,Total net profit,PF,Probability,Max drawdown,Recovery Factor,PayoffRatio,Sharpe Ratio,Total # of trades,# Meses,Net Profit/Mes,Fecha-Inicio,Fecha-Fin,Avg Win,Avg Loss
0,SA_RB001_MNQ_A_T001.csv,RB001,T001,MNQ,5680.5,1.23,14.55,-6111.8,0.929432,7.232887,0.26,550.0,5.913272,960.635667,2026-01-01,2026-06-30,378.28,-52.3


In [197]:
import pytz # Para manejar zonas horarias, movido aquí
from google.colab import auth # Para la autenticación en Colab

# Celda 8 - Limpieza y reordenamiento de datos finales


# Check if consolidated_df exists. If not, try to reconstruct it from the last saved individual cleaned file.
if 'consolidated_df' not in locals() and 'consolidated_df' not in globals():
    print("Advertencia: 'consolidated_df' no definido en el estado actual del kernel. Intentando cargar el último archivo limpio individual.")
    try:
        # Assuming bot_name, test_number, and CLEAN_DATA_PATH are defined in previous cells and are accessible.
        # Corregido: Usar el formato de nombre de archivo consistente con Celda 5.
        cleaned_individual_file_name = f"{bot_name}{test_number}-Limpio.csv"
        individual_output_path = os.path.join(CLEAN_DATA_PATH, cleaned_individual_file_name)

        if os.path.exists(individual_output_path):
            consolidated_df = pd.read_csv(individual_output_path)
            print(f"Éxito: 'consolidated_df' cargado desde {individual_output_path}.")
        else:
            print(f"Error: No se pudo cargar 'consolidated_df'. El archivo '{individual_output_path}' no existe. Por favor, asegúrese de ejecutar la 'Celda 6' primero.")
            consolidated_df = pd.DataFrame() # Create an empty DataFrame to prevent subsequent errors
    except NameError as e:
        print(f"Error de variable al intentar cargar 'consolidated_df': {e}. Asegúrese de que 'bot_name', 'test_number' y 'CLEAN_DATA_PATH' estén definidos en celdas anteriores.")
        consolidated_df = pd.DataFrame() # Create an empty DataFrame as a last resort
    except Exception as e:
        print(f"Error inesperado al cargar 'consolidated_df': {e}")
        consolidated_df = pd.DataFrame() # Create an empty DataFrame as a last resort


# Proceed only if consolidated_df is not empty after the potential loading attempt
if not consolidated_df.empty:
    # Renombrar columnas para que coincidan exactamente con la solicitud del usuario
    # Make a copy to avoid SettingWithCopyWarning later, especially during column renames
    consolidated_df = consolidated_df.copy().rename(columns={
        'Fecha-Inicio': 'Fecha Inicio',
        'Fecha-Fin': 'Fecha Fin',
        'Max drawdown': 'Max drawdown',
        'Net Profit/Mes': 'Net Profit/Mes',
        'Total net profit': 'Total net profit',
        'Probability': 'Percent Profitable' # Renombrar 'Probability' a 'Percent Profitable'
    })

    # Definir el orden de las columnas solicitado por el usuario, incluyendo 'Robot Base' y 'Test ID'
    column_order = [
        'Fecha Inicio',
        'Fecha Fin',
        'Instrumento',
        'Robot Base',
        'Test ID',
        '# Meses',
        'Total # of trades',
        'Total net profit',
        'Net Profit/Mes',
        'PF',
        'Percent Profitable', # CAMBIO: 'Probability' ahora es 'Percent Profitable'
        'Max drawdown',
        'Sharpe Ratio',
        'Recovery Factor',
        'PayoffRatio',
        'Avg Win',
        'Avg Loss'
    ]

    # Seleccionar y reordenar las columnas del DataFrame
    # Filter column_order to only include columns actually present in consolidated_df
    actual_columns_in_order = [col for col in column_order if col in consolidated_df.columns]
    final_df = consolidated_df[actual_columns_in_order]

    print("\n--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---")
    display(final_df.head())
else:
    final_df = pd.DataFrame() # Ensure final_df is defined even if consolidated_df is empty
    print("No se pudo generar 'final_df' porque 'consolidated_df' está vacío o no se pudo cargar.")


--- DataFrame Final con Columnas Limpias y Reordenadas (primeras 5 filas): ---


,Fecha Inicio,Fecha Fin,Instrumento,Robot Base,Test ID,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Percent Profitable,Max drawdown,Sharpe Ratio,Recovery Factor,PayoffRatio,Avg Win,Avg Loss
0,2026-01-01,2026-06-30,MNQ,RB001,T001,5.913272,550.0,5680.5,960.635667,1.23,14.55,-6111.8,0.26,0.929432,7.232887,378.28,-52.3


In [198]:
# Celda 8.1 - Verificar tipos de datos del DataFrame final
print("\n--- Tipos de datos del DataFrame final: ---")
display(final_df.dtypes)


--- Tipos de datos del DataFrame final: ---


,0
Fecha Inicio,object
Fecha Fin,object
Instrumento,object
Robot Base,object
Test ID,object
# Meses,float64
Total # of trades,float64
Total net profit,float64
Net Profit/Mes,float64
PF,float64


In [199]:
import pytz # Para manejar zonas horarias, movido aquí
from google.colab import auth # Para la autenticación en Colab

# Celda 9 -  Define la ruta base para la Tabla Maestra

BASE_MASTER_TABLE_DIR = '/content/drive/MyDrive/Robots/Mis Bots/'
BASE_MASTER_TABLE_NAME = 'Tabla Maestra Bots'

# Helper function to extract Robot and Test ID from a string, designed to handle various formats
def parse_robot_test_id(s):
    if pd.isna(s):
        return None, None
    s = str(s).strip()

    robot_extracted = None
    test_id_extracted = None

    # Try to extract Robot (e.g., RB001)
    match_robot = re.search(r'RB(\d+)', s, re.IGNORECASE)
    if match_robot:
        robot_extracted = f"RB{match_robot.group(1).zfill(3)}" # Ensure 3 digits, e.g., RB001

    # Try to extract Test ID (e.g., T001)
    match_test_id = re.search(r'T(\d+)', s, re.IGNORECASE)
    if match_test_id:
        test_id_extracted = f"T{match_test_id.group(1).zfill(3)}" # Ensure 3 digits, e.g., T001

    return robot_extracted, test_id_extracted


# Renombrar columnas en final_df antes de cualquier otra operación para asegurar consistencia
# Usamos .copy() para evitar SettingWithCopyWarning
# CAMBIO: Renombrar 'Probability' a 'Percent Profitable' en el DataFrame que se va a añadir
final_df_to_add = final_df.rename(columns={'Robot Base': 'Robot', 'Test ID': 'Numero del Test', 'Probability': 'Percent Profitable'}).copy()

# Añadir la columna 'Comentarios' con valores vacíos al DataFrame que se va a añadir
# Esto asegura que la columna exista y no sea sobrescrita por lógica del notebook.
# Este valor será sobrescrito si existe un comentario anterior para este robot/test.
final_df_to_add['Comentarios'] = ''

# Eliminar la línea que añade 'Calificación' con valores nulos iniciales.
# La columna de Calificación (numérica) se añadirá y renombrará en Celda 10.
# final_df_to_add['Calificación'] = pd.NA


# Asegurar que las columnas clave sean de tipo string para la comparación
final_df_to_add['Robot'] = final_df_to_add['Robot'].astype(str)
final_df_to_add['Numero del Test'] = final_df_to_add['Numero del Test'].astype(str)

# Convertir las columnas de fecha en final_df_to_add a tipo datetime para compatibilidad
for col in ['Fecha Inicio', 'Fecha Fin']:
    if col in final_df_to_add.columns:
        final_df_to_add.loc[:, col] = pd.to_datetime(final_df_to_add[col], errors='coerce')


# Construct the path to the most recent master table for loading
# We'll look for any CSV starting with BASE_MASTER_TABLE_NAME
master_files = [f for f in os.listdir(BASE_MASTER_TABLE_DIR) if f.startswith(BASE_MASTER_TABLE_NAME) and f.endswith('.csv')]
master_files.sort(reverse=True) # Sort to get the most recent file first

current_master_table_path = None
if master_files:
    current_master_table_path = os.path.join(BASE_MASTER_TABLE_DIR, master_files[0])


# Verificar si el archivo de la tabla maestra existe
if current_master_table_path and os.path.exists(current_master_table_path):
    print(f"Cargando tabla maestra más reciente desde: {current_master_table_path}")
    master_df = pd.read_csv(current_master_table_path)

    # IMPORTE: Renombrar columnas antiguas en master_df para que coincidan con los nuevos nombres estandarizados
    master_df = master_df.rename(columns={
        '# Trades': 'Total # of trades',
        'NetProfit': 'Total net profit',
        'DD Max': 'Max drawdown',
        'WR': 'Probability', # Mantener para compatibilidad con archivos antiguos si 'WR' existía
        'Probability': 'Percent Profitable' # Renombrar 'Probability' a 'Percent Profitable'
    })

    # --- START: Robust standardization of 'Robot' and 'Numero del Test' in loaded master_df ---
    # First, ensure 'Robot' and 'Numero del Test' columns exist, possibly from old names
    if 'Robot Base' in master_df.columns and 'Robot' not in master_df.columns:
        master_df = master_df.rename(columns={'Robot Base': 'Robot'})
    if 'Test ID' in master_df.columns and 'Numero del Test' not in master_df.columns:
        master_df = master_df.rename(columns={'Test ID': 'Numero del Test'})

    # If the columns still don't exist, create them with placeholder
    if 'Robot' not in master_df.columns:
        master_df['Robot'] = pd.NA
    if 'Numero del Test' not in master_df.columns:
        master_df['Numero del Test'] = pd.NA

    # Apply robust parsing to standardize Robot and Numero del Test
    standardized_robot_col = []
    standardized_test_id_col = []

    for idx, row in master_df.iterrows():
        # Get current values, try to use existing if they seem valid
        current_robot_val = row['Robot']
        current_test_id_val = row['Numero del Test']

        # Attempt to parse from current 'Robot' value
        parsed_r_from_robot, parsed_t_from_robot = parse_robot_test_id(current_robot_val)
        # Attempt to parse from current 'Numero del Test' value
        parsed_r_from_test_id, parsed_t_from_test_id = parse_robot_test_id(current_test_id_val)
        # Attempt to parse from 'Archivo' column if it exists and current keys are problematic
        parsed_r_from_file = None
        parsed_t_from_file = None
        if 'Archivo' in master_df.columns and (pd.isna(current_robot_val) or pd.isna(current_test_id_val) or not str(current_robot_val).startswith('RB') or not str(current_robot_val).startswith('T')):
             parsed_r_from_file, parsed_t_from_file = parse_robot_test_id(row['Archivo'])


        # Prioritize values that look correct (e.g., start with 'RB'/'T')
        # Combine parsed results, prioritizing from specific columns or more complete sources
        final_robot = None
        if parsed_r_from_robot and parsed_r_from_robot.startswith('RB'):
            final_robot = parsed_r_from_robot
        elif parsed_r_from_test_id and parsed_r_from_test_id.startswith('RB'):
            final_robot = parsed_r_from_test_id
        elif parsed_r_from_file and parsed_r_from_file.startswith('RB'):
            final_robot = parsed_r_from_file

        final_test_id = None
        if parsed_t_from_test_id and parsed_t_from_test_id.startswith('T'):
            final_test_id = parsed_t_from_test_id
        elif parsed_t_from_robot and parsed_t_from_robot.startswith('T'):
            final_test_id = parsed_t_from_robot
        elif parsed_t_from_file and parsed_t_from_file.startswith('T'):
            final_test_id = parsed_t_from_file

        standardized_robot_col.append(final_robot if final_robot else 'UNKNOWN_ROBOT')
        standardized_test_id_col.append(final_test_id if final_test_id else 'UNKNOWN_TEST')

    master_df['Robot'] = standardized_robot_col
    master_df['Numero del Test'] = standardized_test_id_col

    # Ensure 'Robot' and 'Numero del Test' are string type after standardization
    master_df['Robot'] = master_df['Robot'].astype(str)
    master_df['Numero del Test'] = master_df['Numero del Test'].astype(str)
    # --- END: Robust standardization of 'Robot' and 'Numero del Test' in loaded master_df ---

    # --- ADDED: Clean up potentially erroneous columns from loaded master_df BEFORE merging ---
    # Drop 'NetProfit/Mes' (without space) if 'Net Profit/Mes' (with space) exists, as the latter is correct.
    if 'NetProfit/Mes' in master_df.columns and 'Net Profit/Mes' in master_df.columns:
        print("Detectado y eliminando columna 'NetProfit/Mes' (sin espacio) de la tabla maestra cargada.")
        master_df = master_df.drop(columns=['NetProfit/Mes'])
    # Rename 'NetProfit/Mes' (without space) to 'Net Profit/Mes' (with space) if only the former exists.
    elif 'NetProfit/Mes' in master_df.columns and 'Net Profit/Mes' not in master_df.columns:
        print("Renombrando columna 'NetProfit/Mes' a 'Net Profit/Mes' en la tabla maestra cargada.")
        master_df = master_df.rename(columns={'NetProfit/Mes': 'Net Profit/Mes'})

    # Drop 'Calificación' if it exists in the loaded master_df, as it will be recalculated later.
    # This is to avoid a 'Calificación' column with NaNs from the previous run.
    if 'Calificación' in master_df.columns:
        print("Eliminando columna 'Calificación' de la tabla maestra cargada para recalcularla.")
        master_df = master_df.drop(columns=['Calificación'])

    # Add 'Comentarios' column to master_df if it doesn't exist, initialized with empty strings
    if 'Comentarios' not in master_df.columns:
        master_df['Comentarios'] = ''

    # --- MODIFICADO: Recuperar y COMBINAR el comentario existente ---
    combined_comment_for_current_run = ''
    if not final_df_to_add.empty:
        current_robot_id = final_df_to_add['Robot'].iloc[0]
        current_test_id = final_df_to_add['Numero del Test'].iloc[0]

        # Find if this specific (robot, test) pair exists in the loaded master_df
        matching_rows = master_df[(master_df['Robot'] == current_robot_id) & (master_df['Numero del Test'] == current_test_id)]

        existing_comment = ''
        if not matching_rows.empty and 'Comentarios' in matching_rows.columns:
            retrieved_comment = matching_rows['Comentarios'].iloc[0]
            if pd.notna(retrieved_comment):
                existing_comment = str(retrieved_comment).strip()

        # Get the new comment from the widget (defined in the previous step)
        # Ensure comment_for_current_run is accessible and defaults to empty string if not set
        new_comment_from_widget = globals().get('comment_for_current_run', '').strip()

        if existing_comment and new_comment_from_widget:
            combined_comment_for_current_run = f"{existing_comment}. {new_comment_from_widget}"
        elif existing_comment:
            combined_comment_for_current_run = existing_comment
        elif new_comment_from_widget:
            combined_comment_for_current_run = new_comment_from_widget

    # Asignar el comentario combinado a la fila que se va a añadir
    if not final_df_to_add.empty:
        final_df_to_add.loc[:, 'Comentarios'] = combined_comment_for_current_run
    # --- FIN MODIFICACIÓN DE COMBINACIÓN DE COMENTARIOS ---

    # --- END ADDED CLEANUP ---

    # Convertir las columnas de fecha en master_df a tipo datetime si es necesario
    for col in ['Fecha Inicio', 'Fecha Fin']:
        if col in master_df.columns:
            # Convert to string first to avoid errors with mixed types, then to datetime
            master_df[col] = master_df[col].astype(str)
            master_df.loc[:, col] = pd.to_datetime(master_df[col], errors='coerce')


    print("\n--- Tabla Maestra Original (primeras 5 filas estandarizadas y limpiadas): ---")
    display(master_df.head())

    # Get all unique (Robot, Numero del Test) pairs from the current processed data
    processed_keys = final_df_to_add[['Robot', 'Numero del Test']].drop_duplicates()

    # Create a boolean mask to identify rows in master_df that should be removed
    # These are rows whose (Robot, Numero del Test) pair is present in processed_keys
    mask_to_remove = master_df.set_index(['Robot', 'Numero del Test']).index.isin(
        processed_keys.set_index(['Robot', 'Numero del Test']).index
    )

    # Filter master_df to remove rows that are present in the current processed data
    master_df_filtered = master_df[~mask_to_remove].copy()

    # Concatenate the filtered master_df with the new records
    updated_master_df = pd.concat([master_df_filtered, final_df_to_add], ignore_index=True)

else:
    print(f"Advertencia: No se encontró una tabla maestra existente. Creando una nueva tabla maestra con los datos actuales.")
    updated_master_df = final_df_to_add.copy() # La primera entrada será la ejecución actual

# --- NUEVA LÓGICA DE LIMPIEZA DE COLUMNAS SIMILARES (retained for final check) ---
# This block acts as a fallback to ensure that after concatenation,
# if 'NetProfit/Mes' (without space) is still present and 'Net Profit/Mes' (with space) also exists,
# the former is removed. This might catch issues if final_df_to_add somehow introduced it, though unlikely.
if 'NetProfit/Mes' in updated_master_df.columns and 'Net Profit/Mes' in updated_master_df.columns:
    print("Detectado y eliminando columna 'NetProfit/Mes' (sin espacio) duplicada en el DataFrame maestro actualizado.")
    updated_master_df = updated_master_df.drop(columns=['NetProfit/Mes'])
elif 'NetProfit/Mes' in updated_master_df.columns and 'Net Profit/Mes' not in updated_master_df.columns:
    print("Renombrando columna 'NetProfit/Mes' a 'Net Profit/Mes' en el DataFrame maestro actualizado.")
    updated_df = updated_master_df.rename(columns={'NetProfit/Mes': 'Net Profit/Mes'})
# --- FIN NUEVA LÓGICA DE LIMPIEZA ---


# Reordenar columnas para colocar 'Robot' y 'Numero del Test' al principio
expected_cols_order = [
    'Robot', 'Numero del Test', 'Fecha Inicio', 'Fecha Fin', 'Instrumento',
    '# Meses', 'Total # of trades', 'Total net profit', 'Net Profit/Mes', 'PF', 'Percent Profitable', # CAMBIO: 'Probability' ahora es 'Percent Profitable'
    'Max drawdown',
    'Sharpe Ratio',
    'Recovery Factor', 'PayoffRatio', 'Avg Win', 'Avg Loss',
    'Calificación',
    'Comentarios'
]
# Add any missing columns to updated_master_df that are in expected_cols_order, filling with NaN
for col in expected_cols_order:
    if col not in updated_master_df.columns:
        updated_master_df[col] = pd.NA

# Reorder columns based on expected_cols_order
updated_master_df = updated_master_df[expected_cols_order]

# Sort the DataFrame by 'Robot' and 'Numero del Test' as requested by the user
updated_master_df = updated_master_df.sort_values(by=['Robot', 'Numero del Test']).reset_index(drop=True)


print("\n--- Tabla Maestra Actualizada (primeras 5 filas incluyendo los nuevos datos): ---")
display(updated_master_df.head())

# Define the Miami timezone
miami_timezone = pytz.timezone('America/New_York')

# Guardar la tabla maestra actualizada con la fecha y hora actual en la zona horaria de Miami
current_date_str = datetime.now(miami_timezone).strftime('%Y-%m-%d_%H-%M-%S')
MASTER_TABLE_PATH_DATED = os.path.join(BASE_MASTER_TABLE_DIR, f"{BASE_MASTER_TABLE_NAME}_{current_date_str}.csv")

# --- NUEVO: Eliminar archivos de tabla maestra anteriores ---
print("\nEliminando versiones anteriores de la tabla maestra...")
for f_name in os.listdir(BASE_MASTER_TABLE_DIR):
    if f_name.startswith(BASE_MASTER_TABLE_NAME) and f_name.endswith('.csv'):
        file_path_to_delete = os.path.join(BASE_MASTER_TABLE_DIR, f_name)
        try:
            os.remove(file_path_to_delete)
            print(f"  - Eliminado: {f_name}")
        except Exception as e:
            print(f"  - Error al eliminar {f_name}: {e}")
# --- FIN NUEVO ---

updated_master_df.to_csv(MASTER_TABLE_PATH_DATED, index=False)
print(f"\nTabla maestra actualizada y guardada en: {MASTER_TABLE_PATH_DATED}")

Cargando tabla maestra más reciente desde: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots_2026-08-22_22-03-40.csv
Eliminando columna 'Calificación' de la tabla maestra cargada para recalcularla.

--- Tabla Maestra Original (primeras 5 filas estandarizadas y limpiadas): ---


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Percent Profitable,Max drawdown,Sharpe Ratio,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Caracterización,Actividad Sugerida,Comentarios
0,RB001,T001,2026-01-01 00:00:00,2026-06-30 00:00:00,MNQ,5.913272,550.0,5680.5,960.635667,1.23,14.55,-6111.8,0.26,0.929432,7.232887,378.28,-52.3,"Revisar a fondo lógica, parámetros o instrumento",Descartar o rediseñar completamente,NaN



--- Tabla Maestra Actualizada (primeras 5 filas incluyendo los nuevos datos): ---


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Percent Profitable,Max drawdown,Sharpe Ratio,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Calificación,Comentarios
0,RB001,T001,2026-01-01 00:00:00,2026-06-30 00:00:00,MNQ,5.913272,550.0,5680.5,960.635667,1.23,14.55,-6111.8,0.26,0.929432,7.232887,378.28,-52.3,<NA>,



Eliminando versiones anteriores de la tabla maestra...
  - Eliminado: Tabla Maestra Bots_2026-08-22_22-03-40.csv

Tabla maestra actualizada y guardada en: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots_2026-08-22_22-06-23.csv


In [200]:
# Celda 9.1 - Verificar datos

# Define the base path for the master table (same as in Celda 9)
BASE_MASTER_TABLE_DIR = '/content/drive/MyDrive/Robots/Mis Bots/'
BASE_MASTER_TABLE_NAME = 'Tabla Maestra Bots'

MASTER_TABLE_PATH = None

# Always re-scan the directory and find the most recent master table file
master_files = [f for f in os.listdir(BASE_MASTER_TABLE_DIR) if f.startswith(BASE_MASTER_TABLE_NAME) and f.endswith('.csv')]
master_files.sort(reverse=True) # Sort to get the most recent file first

if master_files:
    MASTER_TABLE_PATH = os.path.join(BASE_MASTER_TABLE_DIR, master_files[0])


if MASTER_TABLE_PATH:
    print(f"Verificando el contenido de la tabla maestra más reciente: {MASTER_TABLE_PATH}")
    if os.path.exists(MASTER_TABLE_PATH):
        verified_master_df = pd.read_csv(MASTER_TABLE_PATH)

        # Mostrar el DataFrame actualizado con todos los indicadores
        print("Tabla Maestra Actualizada con todos los Indicadores y Calificaciones:")
        display(verified_master_df)

        print(f"Total de filas en la tabla maestra: {len(verified_master_df)}")
    else:
        print(f"Error: El archivo '{MASTER_TABLE_PATH}' no existe o no se pudo cargar.")
else:
    print(f"El archivo de la Tabla Maestra (con prefijo '{BASE_MASTER_TABLE_NAME}') no se encontró en '{BASE_MASTER_TABLE_DIR}'.")

Verificando el contenido de la tabla maestra más reciente: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots_2026-08-22_22-06-23.csv
Tabla Maestra Actualizada con todos los Indicadores y Calificaciones:


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Percent Profitable,Max drawdown,Sharpe Ratio,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Calificación,Comentarios
0,RB001,T001,2026-01-01 00:00:00,2026-06-30 00:00:00,MNQ,5.913272,550.0,5680.5,960.635667,1.23,14.55,-6111.8,0.26,0.929432,7.232887,378.28,-52.3,NaN,NaN


Total de filas en la tabla maestra: 1


In [201]:
#Celda 10 - Calificación

import pytz
from datetime import datetime

def calculate_numeric_rating(row, criteria):
    total_weighted_score = 0
    total_applicable_weight = 0

    # Mapeo de nombres de indicadores en rating_criteria a nombres de columna en updated_master_df
    # Este mapeo necesita ser dinámico o al menos exhaustivo para todos los indicadores posibles
    indicator_to_column = {
        'PF': 'PF',
        'Percent profitable': 'Percent Profitable',
        'Recovery Factor': 'Recovery Factor',
        'Sharpe ratio': 'Sharpe Ratio',
        'PayoffRatio': 'PayoffRatio' # Este se añadirá dinámicamente si existe en default_weights
        # Agrega otros mapeos según sea necesario si los nombres en criteria difieren de los de la columna del DataFrame
    }

    # Ensure indicator_to_column includes all keys from criteria and defaults to indicator_name if not explicitly mapped
    for indicator_name_from_criteria in criteria.keys():
        if indicator_name_from_criteria not in indicator_to_column:
            # Asumir que el nombre de la columna es el mismo que el nombre del indicador si no hay un mapeo explícito
            indicator_to_column[indicator_name_from_criteria] = indicator_name_from_criteria

    for indicator_name, detail in criteria.items():
        # Ensure we only process indicators that have a corresponding column name
        if indicator_name not in indicator_to_column:
            continue # Skip if no mapping exists (shouldn't happen with the dynamic update above)

        column_name = indicator_to_column[indicator_name]

        # Si el indicador no tiene una columna mapeada en el df principal o si el indicador no está en el df
        if column_name not in row.index:
            print(f"Advertencia: Indicador '{indicator_name}' con nombre de columna '{column_name}' no encontrado en la fila del DataFrame. Saltando.")
            continue

        robot_value = row.get(column_name)

        if pd.isna(robot_value) or robot_value is None:
            print(f"Advertencia: El valor del indicador '{indicator_name}' es nulo o no está disponible para el robot. Saltando este indicador.")
            continue # No incluir este indicador si su valor falta

        weight = detail['weight']
        score_for_indicator = 0

        if 'ranges' in detail: # Usamos la nueva estructura con múltiples rangos
            value_for_comparison = abs(robot_value) if indicator_name == 'Max drawdown' else robot_value # Se mantiene 'Max drawdown' por si se vuelve a incluir

            # Iterar a través de los rangos (ya están ordenados por puntuación ascendente)
            # Tomaremos la puntuación del rango más alto que coincida
            for r in detail['ranges']:
                current_score = r['score']
                current_range = r['range']

                if current_range is None: # Skip if the range itself couldn't be parsed
                    continue

                # Manejo de rangos categóricos (si se implementan más adelante)
                if isinstance(current_range, dict) and current_range.get('type') == 'category':
                    # Lógica para categorías aquí si es necesario
                    pass # Por ahora no hay categorías activas en los rangos numéricos
                elif isinstance(current_range, tuple) and len(current_range) == 2:
                    # Manejo de rangos numéricos [lower, upper]
                    lower_bound, upper_bound = current_range
                    if lower_bound <= value_for_comparison <= upper_bound:
                        score_for_indicator = current_score # Guardar la puntuación de este rango

            # Si no se encontró ningún rango, la puntuación_para_indicador permanece en 0
        else:
            # Esto es un fallback para el caso de 'Expectancy' o indicadores categóricos si se implementaran así.
            # Por ahora, se asume que todos los indicadores procesados tendrán 'ranges'.
            pass # No asignar puntuación si no hay rangos definidos para este indicador

        total_weighted_score += score_for_indicator * weight
        total_applicable_weight += weight

    if total_applicable_weight == 0:
        return pd.NA # No se pudieron aplicar indicadores o todos los valores estaban ausentes

    final_numeric_score = (total_weighted_score / total_applicable_weight) if total_applicable_weight > 0 else 0
    return final_numeric_score

# Aplicar la función de calificación numérica a cada fila de updated_master_df
updated_master_df['Calificación Numérica'] = updated_master_df.apply(lambda row: calculate_numeric_rating(row, rating_criteria), axis=1)

# Derivar la calificación cualitativa a partir de la numérica (temporalmente)
def get_qualitative_rating(numeric_score):
    if pd.isna(numeric_score):
        return pd.NA
    if numeric_score >= 80:
        return 'Excelente'
    elif numeric_score >= 50:
        return 'Aceptable'
    else:
        return 'Malo'

updated_master_df['Calificación (Cualitativa)'] = updated_master_df['Calificación Numérica'].apply(get_qualitative_rating)

# Derivar la caracterización
def get_caracterizacion(qualitative_rating):
    if pd.isna(qualitative_rating):
        return pd.NA
    if qualitative_rating == 'Excelente':
        return 'Potencialmente fuerte, revisar optimización'
    elif qualitative_rating == 'Aceptable':
        return 'Necesita ajustes, buen punto de partida'
    elif qualitative_rating == 'Malo':
        return 'Revisar a fondo lógica, parámetros o instrumento'
    return pd.NA

updated_master_df['Caracterización'] = updated_master_df['Calificación (Cualitativa)'].apply(get_caracterizacion)

# Derivar la actividad sugerida
def get_actividad_sugerida(qualitative_rating):
    if pd.isna(qualitative_rating):
        return pd.NA
    if qualitative_rating == 'Excelente':
        return 'Continuar monitoreo, considerar escalabilidad'
    elif qualitative_rating == 'Aceptable':
        return 'Analizar indicadores débiles, realizar backtesting incremental'
    elif qualitative_rating == 'Malo':
        return 'Descartar o rediseñar completamente'
    return pd.NA

updated_master_df['Actividad Sugerida'] = updated_master_df['Calificación (Cualitativa)'].apply(get_actividad_sugerida)

# Eliminar la columna de calificación cualitativa, ya que el usuario quiere que 'Calificación' sea la numérica
updated_master_df = updated_master_df.drop(columns=['Calificación (Cualitativa)'], errors='ignore')

# Eliminar cualquier columna 'Calificación' preexistente (que contenga NaNs) antes de renombrarla con el valor numérico.
# Esto es para manejar el caso donde 'Calificación' se añadió en Celda 9 con pd.NA
if 'Calificación' in updated_master_df.columns:
    updated_master_df = updated_master_df.drop(columns=['Calificación'], errors='ignore')

# Renombrar 'Calificación Numérica' a 'Calificación' según la última solicitud del usuario
updated_master_df = updated_master_df.rename(columns={'Calificación Numérica': 'Calificación'})

# Guardar la tabla maestra actualizada con la calificación
# (Se replica la lógica de guardado de Celda 9 para asegurar que la versión final se persista)
miami_timezone = pytz.timezone('America/New_York')
current_date_str = datetime.now(miami_timezone).strftime('%Y-%m-%d_%H-%M-%S')
MASTER_TABLE_PATH_DATED = os.path.join(BASE_MASTER_TABLE_DIR, f"{BASE_MASTER_TABLE_NAME}_{current_date_str}.csv")

# Eliminar archivos de tabla maestra anteriores antes de guardar el nuevo
print("\nEliminando versiones anteriores de la tabla maestra antes de guardar la actualizada...")
for f_name in os.listdir(BASE_MASTER_TABLE_DIR):
    if f_name.startswith(BASE_MASTER_TABLE_NAME) and f_name.endswith('.csv'):
        file_path_to_delete = os.path.join(BASE_MASTER_TABLE_DIR, f_name)
        try:
            os.remove(file_path_to_delete)
            print(f"  - Eliminado: {f_name}")
        except Exception as e:
            print(f"  - Error al eliminar {f_name}: {e}")

updated_master_df.to_csv(MASTER_TABLE_PATH_DATED, index=False)
print(f"\nTabla maestra actualizada (con calificaciones) guardada en: {MASTER_TABLE_PATH_DATED}")

# Reordenar columnas para incluir las nuevas y asegurar el orden final
expected_cols_order = [
    'Robot', 'Numero del Test', 'Fecha Inicio', 'Fecha Fin', 'Instrumento',
    '# Meses', 'Total # of trades', 'Total net profit', 'Net Profit/Mes', 'PF', 'Percent Profitable',
    'Max drawdown',
    'Sharpe Ratio',
    'Recovery Factor', 'PayoffRatio', 'Avg Win', 'Avg Loss',
    'Calificación', # Ahora contiene el promedio ponderado numérico
    'Caracterización', 'Actividad Sugerida',
    'Comentarios'
]

# Add any missing columns to updated_master_df that are in expected_cols_order, filling with NaN
for col in expected_cols_order:
    if col not in updated_master_df.columns:
        updated_master_df[col] = pd.NA

# Reorder columns based on expected_cols_order
updated_master_df = updated_master_df[expected_cols_order]

print("\n--- Tabla Maestra Actualizada y Final (primeras 5 filas con calificación numérica): ---")
display(updated_master_df.head())


Eliminando versiones anteriores de la tabla maestra antes de guardar la actualizada...
  - Eliminado: Tabla Maestra Bots_2026-08-22_22-06-23.csv

Tabla maestra actualizada (con calificaciones) guardada en: /content/drive/MyDrive/Robots/Mis Bots/Tabla Maestra Bots_2026-08-22_22-06-49.csv

--- Tabla Maestra Actualizada y Final (primeras 5 filas con calificación numérica): ---


,Robot,Numero del Test,Fecha Inicio,Fecha Fin,Instrumento,# Meses,Total # of trades,Total net profit,Net Profit/Mes,PF,Percent Profitable,Max drawdown,Sharpe Ratio,Recovery Factor,PayoffRatio,Avg Win,Avg Loss,Calificación,Caracterización,Actividad Sugerida,Comentarios
0,RB001,T001,2026-01-01 00:00:00,2026-06-30 00:00:00,MNQ,5.913272,550.0,5680.5,960.635667,1.23,14.55,-6111.8,0.26,0.929432,7.232887,378.28,-52.3,41.0,"Revisar a fondo lógica, parámetros o instrumento",Descartar o rediseñar completamente,


In [202]:
print("Indicadores cargados de tu Google Sheet ('rating_table_df['Indicador']'):")
display(rating_table_df['Indicador'].tolist())

Indicadores cargados de tu Google Sheet ('rating_table_df['Indicador']'):


['PF',
 'PayoffRatio',
 'Percent profitable',
 'Sharpe ratio',
 'Recovery Factor',
 '',
 '',
 'Fórmula Calificación']

---